# Chapitre 1 — Télécharger les données (clé Kaggle)

Ce notebook **extrait le jeu de données RSNA** via l'API Kaggle. C'est le
prérequis aux chapitres 2.5, 3, 4 et 5.

**Avant de lancer :**
1. Avoir accepté les règles de la compétition sur
   <https://www.kaggle.com/competitions/rsna-breast-cancer-detection/rules>
   (sinon Kaggle renvoie une erreur 403).
2. Les identifiants Kaggle doivent être disponibles : `docker-run.sh` monte
   `~/.kaggle` (avec `kaggle.json`) en lecture seule dans le conteneur. Sinon,
   renseigner `KAGGLE_USERNAME` / `KAGGLE_KEY` (voir `.env.example`).

Le jeu complet (images DICOM) fait **~314 Go** → téléchargement optionnel,
désactivé par défaut. On commence par les labels (`train.csv`, ~2 Mo).

In [1]:
import os
# Vérifie les identifiants AVANT d'importer kaggle (l'import authentifie aussitôt).
os.environ.setdefault('KAGGLE_CONFIG_DIR', os.path.expanduser('~/.kaggle'))
have_json = os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json'))
have_env = bool(os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))
assert have_json or have_env, (
    'Identifiants Kaggle absents : monte ~/.kaggle (kaggle.json) ou renseigne '
    'KAGGLE_USERNAME / KAGGLE_KEY (voir README).')
import kaggle  # déclenche l'authentification
print('Authentification Kaggle OK')

AssertionError: Identifiants Kaggle absents : monte ~/.kaggle (kaggle.json) ou renseigne KAGGLE_USERNAME / KAGGLE_KEY (voir README).

In [ ]:
import glob, zipfile

COMP = 'rsna-breast-cancer-detection'
DATA_DIR = os.path.expanduser('~/data/rsna')   # volume monté -> persistant
os.makedirs(DATA_DIR, exist_ok=True)

def _unzip_all(folder):
    """Décompresse puis supprime tous les .zip d'un dossier."""
    for z in glob.glob(os.path.join(folder, '*.zip')):
        print('décompression', os.path.basename(z))
        with zipfile.ZipFile(z) as zf:
            zf.extractall(folder)
        os.remove(z)

print('Cible :', DATA_DIR)

In [ ]:
# --- Labels seuls (léger) : de quoi explorer avant de tout télécharger ---
import pandas as pd
train_csv = os.path.join(DATA_DIR, 'train.csv')
if not os.path.exists(train_csv):
    kaggle.api.competition_download_file(COMP, 'train.csv', path=DATA_DIR, quiet=False)
    _unzip_all(DATA_DIR)
df = pd.read_csv(train_csv)
print('train.csv :', df.shape)
print('prévalence cancer :', round(df['cancer'].mean(), 4))
df.head()

In [ ]:
# --- Jeu COMPLET (images DICOM, ~314 Go) : passer FULL=True pour télécharger ---
FULL = False
if FULL and not os.path.isdir(os.path.join(DATA_DIR, 'train_images')):
    kaggle.api.competition_download_files(COMP, path=DATA_DIR, quiet=False)
    _unzip_all(DATA_DIR)
    print('Téléchargement complet terminé.')
else:
    present = os.path.isdir(os.path.join(DATA_DIR, 'train_images'))
    print('train_images déjà présent :' if present else 'FULL=False (téléchargement complet ignoré).',
          present)

## 🧪 Smoke test

Vérifie que le code clé de ce chapitre tourne **sans télécharger les données ni lancer le preprocessing complet** (entrées synthétiques).

In [ ]:
# 🧪 Smoke test — valide la logique locale SANS réseau ni clé Kaggle.
import os, glob, zipfile, tempfile

if '_unzip_all' not in dir():                 # permet d'exécuter ce test seul
    def _unzip_all(folder):
        for z in glob.glob(os.path.join(folder, '*.zip')):
            with zipfile.ZipFile(z) as zf:
                zf.extractall(folder)
            os.remove(z)

with tempfile.TemporaryDirectory() as d:
    z = os.path.join(d, 'demo.zip')
    with zipfile.ZipFile(z, 'w') as zf:
        zf.writestr('train.csv', 'patient_id,cancer\n1,0\n2,1\n')
    _unzip_all(d)
    assert os.path.isfile(os.path.join(d, 'train.csv')), 'extraction KO'
    assert not glob.glob(os.path.join(d, '*.zip')), 'le .zip aurait dû être supprimé'

creds = bool(os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json'))
             or (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')))
print('✅ _unzip_all (extraction + nettoyage) OK | identifiants Kaggle présents :', creds)